# AI Engineering: Retrieval-Augmented Generation (RAG)

## Connect LLMs to External Knowledge Bases

**Problem**: LLMs have fixed knowledge; can't access real-time or proprietary data  
**Solution**: RAG - Retrieve relevant documents, then generate answer  
**Impact**: Most deployed LLM pattern in production (2024-2026)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

np.random.seed(42)

## Part 1: RAG Architecture

```
Question → [Retriever] → Top-K Documents → [Reranker] → Context
                                                              ↓
                                                           [LLM] → Answer
```

### Key Components
1. **Encoder**: Convert text to embeddings (semantic search)
2. **Retriever**: Dense similarity search (cosine similarity)
3. **Reranker**: Cross-encoder for better ranking
4. **LLM**: Generate answer given context
5. **Evaluator**: Measure retrieval quality (NDCG, F1, etc.)


In [ ]:
# Create synthetic document corpus
documents = [
    "Python is a programming language used for machine learning and data science.",
    "Machine learning algorithms learn patterns from data without explicit programming.",
    "Deep learning uses neural networks with multiple layers to learn hierarchical features.",
    "Natural language processing helps computers understand and generate human language.",
    "Transformers are neural network architectures that use attention mechanisms.",
    "BERT is a transformer-based model for understanding text meaning.",
    "GPT models generate text by predicting the next token in a sequence.",
    "Embeddings represent text as vectors capturing semantic meaning.",
    "Retrieval-augmented generation combines search with language models.",
    "Vector databases store and search embeddings efficiently.",
]

questions = [
    "What is machine learning?",
    "How do transformers work?",
    "What is RAG?",
]

print(f"Document corpus: {len(documents)} documents")
for i, doc in enumerate(documents[:3]):
    print(f"  {i+1}. {doc}")

In [ ]:
class RAGSystem:
    """Simple RAG implementation using TF-IDF."""
    
    def __init__(self, documents):
        self.documents = documents
        # Vectorize documents
        self.vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
        self.doc_vectors = self.vectorizer.fit_transform(documents)
    
    def retrieve(self, query, k=3):
        """Retrieve top-k documents."""
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.doc_vectors)[0]
        
        top_indices = np.argsort(similarities)[::-1][:k]
        results = [
            {
                'doc_id': idx,
                'text': self.documents[idx],
                'score': similarities[idx]
            }
            for idx in top_indices
        ]
        return results
    
    def generate_answer(self, query, k=3):
        """Retrieve documents and generate mock answer."""
        retrieved = self.retrieve(query, k=k)
        
        # Concatenate context
        context = "\n".join([f"- {r['text']}" for r in retrieved])
        
        # Mock LLM response
        answer = f"Based on the retrieved documents:\n{context}\n\n[LLM would generate answer here]"
        
        return {
            'query': query,
            'retrieved_docs': retrieved,
            'context': context,
            'answer': answer
        }

rag = RAGSystem(documents)

# Test RAG
for question in questions:
    result = rag.generate_answer(question, k=2)
    print(f"\nQuestion: {question}")
    print(f"Retrieved documents:")
    for doc in result['retrieved_docs']:
        print(f"  - Score {doc['score']:.3f}: {doc['text'][:60]}...")

## Part 2: Evaluation Metrics


In [ ]:
def compute_ndcg(relevances, k=5):
    """NDCG: Normalized Discounted Cumulative Gain.
    
    DCG = ∑ (2^rel_i - 1) / log₂(i+1)
    NDCG = DCG / IDCG (ideal DCG with sorted relevances)
    """
    relevances = np.array(relevances)[:k]
    
    # Discounted cumulative gain
    gains = 2 ** relevances - 1
    discounts = np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains / discounts)
    
    # Ideal DCG (perfectly sorted)
    ideal_gains = 2 ** np.sort(relevances)[::-1] - 1
    ideal_discounts = np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = np.sum(ideal_gains / ideal_discounts)
    
    return dcg / idcg if idcg > 0 else 0

def compute_mrr(relevances):
    """MRR: Mean Reciprocal Rank - position of first relevant result."""
    for i, rel in enumerate(relevances):
        if rel > 0:
            return 1 / (i + 1)
    return 0

# Simulate retrieval rankings
relevance_perfect = [1, 1, 1, 0, 0]  # Perfect ranking
relevance_poor = [0, 0, 1, 1, 1]      # Poor ranking
relevance_medium = [1, 0, 1, 0, 1]    # Medium ranking

print("\nRETRIEVAL EVALUATION METRICS")
print("="*60)
print(f"{'Ranking':<20} {'NDCG@5':>15} {'MRR':>15}")
print("-"*60)
for name, rel in [('Perfect', relevance_perfect), 
                   ('Medium', relevance_medium),
                   ('Poor', relevance_poor)]:
    ndcg = compute_ndcg(rel)
    mrr = compute_mrr(rel)
    print(f"{name:<20} {ndcg:>15.4f} {mrr:>15.4f}")

## Part 3: Visualization


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Retrieval scores
ax = axes[0, 0]
result = rag.generate_answer("What is machine learning?", k=5)
scores = [r['score'] for r in result['retrieved_docs']]
ax.barh(range(len(scores)), scores, color='steelblue', alpha=0.7)
ax.set_yticks(range(len(scores)))
ax.set_yticklabels([f"Doc {r['doc_id']}" for r in result['retrieved_docs']])
ax.set_xlabel('Similarity Score')
ax.set_title('Document Retrieval Scores')
ax.grid(True, alpha=0.3, axis='x')

# Plot 2: NDCG vs k
ax = axes[0, 1]
k_values = range(1, 6)
ndcg_scores = [compute_ndcg(relevance_perfect, k) for k in k_values]
ax.plot(k_values, ndcg_scores, 'o-', linewidth=2, markersize=8, color='darkgreen')
ax.set_xlabel('Number of Retrieved Documents (k)')
ax.set_ylabel('NDCG Score')
ax.set_title('NDCG@k: Impact of Retrieval Depth')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.1])

# Plot 3: Ranking comparison
ax = axes[1, 0]
rankings = ['Perfect', 'Medium', 'Poor']
ndcg_vals = [compute_ndcg(rel) for rel in [relevance_perfect, relevance_medium, relevance_poor]]
mrr_vals = [compute_mrr(rel) for rel in [relevance_perfect, relevance_medium, relevance_poor]]

x = np.arange(len(rankings))
width = 0.35
ax.bar(x - width/2, ndcg_vals, width, label='NDCG@5', alpha=0.8, color='steelblue')
ax.bar(x + width/2, mrr_vals, width, label='MRR', alpha=0.8, color='orange')
ax.set_ylabel('Score')
ax.set_title('Metric Comparison Across Rankings')
ax.set_xticks(x)
ax.set_xticklabels(rankings)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

# Plot 4: RAG pipeline stages
ax = axes[1, 1]
stages = ['Query\nEmbedding', 'Dense\nRetrieval', 'Top-K\nSelection', 'LLM\nGeneration']
latencies = [5, 50, 10, 200]  # milliseconds
colors_pipe = ['blue', 'green', 'orange', 'red']

ax.barh(stages, latencies, color=colors_pipe, alpha=0.7, edgecolor='black')
ax.set_xlabel('Latency (ms)')
ax.set_title('RAG Pipeline Latency Breakdown')
for i, lat in enumerate(latencies):
    ax.text(lat, i, f' {lat}ms', va='center')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('SECTION_3_AI_ENGINEERING/rag_system.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRAG SYSTEM PERFORMANCE")
print("="*60)
print(f"Query latency: 5ms (embedding)")
print(f"Retrieval latency: 50ms (searching {len(documents)} docs)")
print(f"Selection latency: 10ms (top-3 reranking)")
print(f"LLM latency: 200ms (token generation)")
print(f"Total latency: 265ms")
print(f"\nRecommended for: Q&A, customer service, knowledge bases")

## Key Insights

1. **Most deployed pattern**: RAG is the gold standard for LLM applications in 2024-2026
2. **Evaluation is critical**: NDCG, MRR measure retrieval quality
3. **Latency tradeoff**: More reranking = better quality but slower
4. **Hybrid approach**: Combine keyword search + dense retrieval

### References
- Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP"
